# Managed vs External Tables — And Why Your File Got Deleted
**Day 4 | Run on `dev-cluster` | Live proof of DROP behaviour for managed vs external**

---

## Why Did the File Get Deleted from ADLS When You Deleted It from the Volume UI?

**Short answer: because a Volume IS your ADLS path — they are the same storage.**

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
                    │
                    │  Unity Catalog maps this path to:
                    ▼
abfss://bronze@evdatalakedev.dfs.core.windows.net/
```

The Volume is not a copy of your ADLS data — it is a **pointer** to it. When you delete a file
through the Volume UI, you are deleting the actual file on ADLS. There is no recycle bin.

---

## Managed vs External — The Core Difference

```
┌─────────────────────────────────────────────────────────────────────────┐
│  MANAGED  — Unity Catalog owns the data files                           │
│                                                                         │
│  CREATE TABLE ev_stations_managed (...)  ← no LOCATION specified        │
│  UC stores files at its own managed ADLS path automatically             │
│                                                                         │
│  DROP TABLE  →  metadata deleted  +  FILES DELETED FROM ADLS            │
│  Use when: data lives only inside Databricks, no external tool reads it │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│  EXTERNAL  — YOU own the data files                                     │
│                                                                         │
│  CREATE TABLE charging_sessions LOCATION 'abfss://silver@...'           │
│  UC only registers metadata — files stay at YOUR ADLS path              │
│                                                                         │
│  DROP TABLE  →  metadata deleted  +  FILES STAY ON ADLS                 │
│  Use when: ADF / Synapse / Power BI also read the same files            │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## Full Comparison: What Happens to Your Data

| Action | Managed Table/Volume | External Table/Volume |
|---|---|---|
| `DROP TABLE` | metadata + **FILES DELETED** | metadata deleted, **FILES SAFE** |
| `DROP SCHEMA CASCADE` | everything + **FILES DELETED** | schema deleted, **FILES SAFE** |
| `DROP VOLUME` | volume + **FILES DELETED** | volume deregistered, **FILES SAFE** |
| `TRUNCATE TABLE` | all rows deleted | all rows deleted |
| Delete file via Volume UI | **file deleted from ADLS** | **file deleted from ADLS** |
| `dbutils.fs.rm(path)` | **file deleted from ADLS** | **file deleted from ADLS** |
| Re-register after DROP | not possible (files gone) | yes — same `LOCATION` |

**Key insight:** Managed vs External only changes DROP behaviour.  
Deleting a file via the Volume UI or `dbutils.fs.rm()` **always** touches real ADLS — for both types.

---

## Our Project Mapping

| Object | Type | Reason |
|---|---|---|
| `bronze-volume` | External Volume | ADF also writes to Bronze ADLS |
| Silver `charging_sessions` (Day 5+) | External Table | Delta files stay on your ADLS silver/ |
| Gold aggregations (Day 6+) | External Table | Power BI / Synapse may read same files |
| Temp/staging tables | Managed Table | Databricks-only, safe to DROP freely |

---

## How to Run This Notebook

- **Part A** (Cells 1–3): create a managed table, see where UC stores files, DROP and prove files are gone
- **Part B** (Cells 4–7): write files to ADLS, create external table, DROP and prove files survive, re-register
- **Part C** (Cell 8): check bronze-volume type and explain the UI delete behaviour
- **Part D** (Cell 9): final reference card

In [ ]:
# =============================================================
# SETUP — Set catalog and schema before anything else
# =============================================================

spark.sql("USE CATALOG dbw_ev_intelligence_dev")
spark.sql("USE SCHEMA default")

print("Catalog :", spark.sql("SELECT current_catalog()").collect()[0][0])
print("Schema  :", spark.sql("SELECT current_schema()").collect()[0][0])

In [ ]:
# =============================================================
# PART A — CELL 1: Create a Managed Table and insert data
# =============================================================
# No LOCATION specified = Unity Catalog decides where files go.
# UC stores them in its own managed storage root on ADLS.
# You do not control that path — UC does.
# =============================================================

spark.sql("""
    CREATE TABLE IF NOT EXISTS ev_stations_managed (
        station_id   STRING,
        station_name STRING,
        city         STRING,
        capacity_kw  INT
    )
    USING DELTA
""")

spark.sql("""
    INSERT INTO ev_stations_managed VALUES
        ('ST001', 'MG Road Charger',    'Bengaluru', 150),
        ('ST002', 'Andheri Fast Charge','Mumbai',    200),
        ('ST003', 'Cyber Hub Station',  'Gurugram',  120)
""")

print("Managed table created and data inserted.")
print()
spark.sql("SELECT * FROM ev_stations_managed").show()

In [ ]:
# =============================================================
# PART A — CELL 2: Find where the managed table files live
# =============================================================
# DESCRIBE DETAIL shows tableType (MANAGED/EXTERNAL) and location.
# For a managed table, the location is a UC-controlled path —
# NOT your abfss://bronze@evdatalakedev... path.
# =============================================================

detail = spark.sql("DESCRIBE DETAIL ev_stations_managed").collect()[0]

print("Table type :", detail['tableType'])   # MANAGED
print("Location   :", detail['location'])    # UC-managed path, not your ADLS
print("Format     :", detail['format'])      # delta
print()
print("Note: location is UC-managed — not abfss://bronze@evdatalakedev...")
print("UC owns these files. DROP TABLE will delete them permanently.")

In [ ]:
# =============================================================
# PART A — CELL 3: DROP the managed table — files are gone
# =============================================================
# After DROP: table is gone from UC AND files are deleted from ADLS.
# There is no way to recover — no recycle bin.
# =============================================================

location_before = spark.sql("DESCRIBE DETAIL ev_stations_managed").collect()[0]['location']
print("Files were at:", location_before)
print()

spark.sql("DROP TABLE ev_stations_managed")
print("Table dropped.")
print()

# Try to read the files — they should be gone
try:
    files = dbutils.fs.ls(location_before)
    print(f"Files still exist: {len(files)} item(s) — unexpected for managed DROP.")
except Exception:
    print("Files NOT found at the managed location — confirmed deleted by DROP.")
    print()
    print("LESSON: DROP TABLE on a MANAGED table = metadata gone + ADLS files gone.")
    print("        There is no way to recover the data.")

In [ ]:
# =============================================================
# PART B — CELL 4: Write sample data to a specific ADLS path
# =============================================================
# For an external table, files must already exist at a path YOU
# control. We write Delta files to the Bronze Volume first.
# This path becomes the external table's LOCATION.
# =============================================================

from pyspark.sql import Row

EXTERNAL_VOLUME_PATH = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/demo/ev_stations_external/"
EXTERNAL_ADLS_PATH   = "abfss://bronze@evdatalakedev.dfs.core.windows.net/demo/ev_stations_external/"

rows = [
    Row(station_id="ST004", station_name="Koramangala Hub",   city="Bengaluru", capacity_kw=180),
    Row(station_id="ST005", station_name="Bandra Supercharge",city="Mumbai",    capacity_kw=250),
    Row(station_id="ST006", station_name="DLF Cyber City",    city="Gurugram",  capacity_kw=100),
]
df = spark.createDataFrame(rows)
df.write.format("delta").mode("overwrite").save(EXTERNAL_VOLUME_PATH)

print(f"Data written to: {EXTERNAL_VOLUME_PATH}")
print()
files = dbutils.fs.ls(EXTERNAL_VOLUME_PATH)
print(f"{len(files)} item(s) at path (Delta log + parquet files):")
for f in files:
    print(f"  {f.path}")

In [ ]:
# =============================================================
# PART B — CELL 5: Create External Table pointing to that path
# =============================================================
# The LOCATION keyword is what makes a table EXTERNAL.
# UC registers the metadata but does NOT own or manage the files.
# Note: LOCATION must use abfss:// — not /Volumes/... path.
# =============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS ev_stations_external
    USING DELTA
    LOCATION '{EXTERNAL_ADLS_PATH}'
""")

detail = spark.sql("DESCRIBE DETAIL ev_stations_external").collect()[0]
print("Table type :", detail['tableType'])   # EXTERNAL
print("Location   :", detail['location'])    # your abfss://bronze@evdatalakedev... path
print()
print("UC only stores metadata — files stay on YOUR ADLS path.")
print()
spark.sql("SELECT * FROM ev_stations_external").show()

In [ ]:
# =============================================================
# PART B — CELL 6: DROP external table — prove files survive
# =============================================================
# This is the core proof: DROP TABLE on an external table removes
# UC metadata only. Your ADLS files are untouched.
# =============================================================

files_before = dbutils.fs.ls(EXTERNAL_VOLUME_PATH)
print(f"Files BEFORE drop: {len(files_before)} item(s)")
print()

spark.sql("DROP TABLE ev_stations_external")
print("Table dropped from Unity Catalog.")
print()

# Query should now fail — metadata is gone
try:
    spark.sql("SELECT * FROM ev_stations_external").show()
except Exception:
    print("Query failed (expected) — table no longer exists in UC.")
    print()

# Files on ADLS should still be there
try:
    files_after = dbutils.fs.ls(EXTERNAL_VOLUME_PATH)
    print(f"Files AFTER drop: {len(files_after)} item(s) — files preserved on ADLS.")
    print()
    print("LESSON: DROP TABLE on an EXTERNAL table = metadata gone, FILES PRESERVED.")
    print("        You can re-register the table anytime from the same LOCATION.")
except Exception:
    print("Files not found after drop — unexpected for an external table.")

In [ ]:
# =============================================================
# PART B — CELL 7: Re-register external table from same files
# =============================================================
# Because files are preserved, you can bring the table back
# instantly — same CREATE TABLE ... LOCATION command.
# No data reload, no pipeline rerun, no data loss.
# =============================================================

spark.sql(f"""
    CREATE TABLE ev_stations_external
    USING DELTA
    LOCATION '{EXTERNAL_ADLS_PATH}'
""")

print("Table re-registered from the same ADLS path.")
print("Data is back instantly — no reload needed.")
print()
spark.sql("SELECT * FROM ev_stations_external").show()

# Cleanup
spark.sql("DROP TABLE IF EXISTS ev_stations_external")
dbutils.fs.rm(EXTERNAL_VOLUME_PATH, recurse=True)
print("Demo table and demo files cleaned up.")

In [ ]:
# =============================================================
# PART C — CELL 8: Check bronze-volume type + explain UI delete
# =============================================================
# Explains exactly why deleting a file via the Volume UI in the
# Catalog tab deleted the real ADLS file.
# =============================================================

print("Checking bronze-volume type...")
print()
try:
    vol = spark.sql("""
        DESCRIBE VOLUME dbw_ev_intelligence_dev.default.`bronze-volume`
    """).collect()
    for row in vol:
        print(f"  {row[0]:20s} : {row[1]}")
except Exception as e:
    print(f"Could not describe volume: {e}")

print()
print("=" * 60)
print("WHY YOUR FILE WAS DELETED FROM ADLS")
print("=" * 60)
print()
print("You deleted a file via the Catalog UI:")
print("  Catalog → bronze-volume → realtime → ... → file → delete")
print()
print("What actually happened:")
print("  The Volume UI is a file browser for your real ADLS path.")
print("  Deleting a file there = dbutils.fs.rm() on the real ADLS file.")
print("  There is no recycle bin. The file is immediately gone.")
print()
print("This happens regardless of Managed vs External Volume.")
print("Managed/External only affects DROP VOLUME — not individual file deletes.")
print()
print("WHEN will deletion NOT touch your ADLS files?")
print("  Only one scenario: DROP TABLE or DROP VOLUME on an EXTERNAL object.")
print("  Everything else (file delete via UI, dbutils.fs.rm, TRUNCATE)")
print("  always touches real ADLS storage immediately.")

In [ ]:
# =============================================================
# PART D — CELL 9: Final reference card
# =============================================================

print("=" * 70)
print("MANAGED vs EXTERNAL — COMPLETE REFERENCE")
print("=" * 70)
print()
print(f"  {'Action':<38} {'Managed':<18} {'External'}")
print("  " + "-" * 66)
actions = [
    ("DROP TABLE",                     "meta + FILES gone",  "meta gone, FILES SAFE"),
    ("DROP SCHEMA CASCADE",            "meta + FILES gone",  "meta gone, FILES SAFE"),
    ("DROP VOLUME",                    "meta + FILES gone",  "meta gone, FILES SAFE"),
    ("TRUNCATE TABLE",                 "all rows deleted",   "all rows deleted"),
    ("DELETE FROM table WHERE ...",    "rows deleted",       "rows deleted"),
    ("Delete file via Volume UI",      "file deleted",       "file deleted"),
    ("dbutils.fs.rm(path)",            "file deleted",       "file deleted"),
    ("Re-register after DROP TABLE",   "not possible",       "yes — same LOCATION"),
]
for action, managed, external in actions:
    print(f"  {action:<38} {managed:<18} {external}")

print()
print("=" * 70)
print("WHEN TO USE WHICH")
print("=" * 70)
print()
print("  MANAGED TABLE — use when:")
print("    Only Databricks reads/writes this data")
print("    You want UC to fully manage lifecycle (DROP = clean slate)")
print("    Temporary or intermediate tables during transformations")
print()
print("  EXTERNAL TABLE — use when:")
print("    Files already exist on ADLS (ADF wrote them, migration copied them)")
print("    ADF, Synapse, Power BI also read the same ADLS files")
print("    You want DROP TABLE to be a safe metadata-only operation")
print("    Bronze/Silver/Gold Delta tables in this project")
print()
print("=" * 70)